In [ ]:
import os
import pandas as pd

# 1) READ MULTIPLE CSV FILES AND EXTRACT PZ, PX
downloads_folder = r"C:\Users\TrevorWhite\Downloads"
file_names = [
    "MW_PBP_24.csv",
    "WCC_PBP_24.csv",
    "SEC_PBP_24.csv",
    "P12_PBP_24.csv",
    "B10_PBP_24.csv",
    "BE_PBP_24.csv",
    "B12_PBP_24.csv",
    "ACC_PBP_24.csv"
]

data_list = []
for file_name in file_names:
    file_path = os.path.join(downloads_folder, file_name)
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        data_list.append(df_temp)
        print(f"Successfully read: {file_name}")
    else:
        print(f"File not found: {file_name}")

combined_df = pd.concat(data_list, ignore_index=True)
print("All files successfully appended into a single DataFrame.")
print("Combined DataFrame shape:", combined_df.shape)

# List of columns to select (only uniqPitchId, PX, PZ)
pz_px_columns = ["uniqPitchId", "PZ", "PX", "batterHand"]
selected_columns_df = combined_df[pz_px_columns].copy()
print(selected_columns_df.head())

# 2) LOAD TRAINING DATA THAT NEEDS PX, PZ MERGED IN
train_data_path = r"C:\Users\TrevorWhite\Downloads\NCAA_STUFF_PLUS_24_TRAIN.csv"
train_df = pd.read_csv(train_data_path)
print("Train Data shape:", train_df.shape)
print(train_df.head())

# 3) MERGE PX, PZ INTO THE TRAIN DATA ON 'uniqPitchId'
# If 'uniqPitchId' is unique in selected_columns_df, it will attach PZ, PX accordingly.
df = pd.merge(
    train_df,
    selected_columns_df,
    on='uniqPitchId',
    how='left'   # or 'inner' if you only want matching rows
)

print("Merged DataFrame shape:", merged_df.shape)
merged_df.head()

# You can now work with 'merged_df' which contains all columns from train_df plus PZ and PX.


In [ ]:
train_df[train_df['is_fastball'] == True].head(111)

In [ ]:
# Set pandas display options to show all data
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns 
pd.set_option('display.width', None)  # Width of the display in characters
pd.set_option('display.max_colwidth', None)  # Show full content of each column


In [ ]:
# # Check unique levels of pitcher_hand
# unique_pitcher_hands = merged_df['pitchResult'].unique()
# print(unique_pitcher_hands)

# Sort merged_df by pitchResult and display first few rows
df[df['pitchResult'].str.contains('hit', case=False, na=False)].sort_values('pitchResult').head()







In [ ]:
# # ... existing code ...

# # Add these pandas display options at the beginning of your notebook, after the imports
# pd.set_option('display.max_rows', None)  # Show all rows
# pd.set_option('display.max_columns', None)  # Show all columns
# pd.set_option('display.width', None)  # Width of the display in characters
# pd.set_option('display.max_colwidth', None)  # Show full content of each column

# Create the 'same_side' column
df['same_side'] = np.where(merged_df['batterHand'] == merged_df['pitcher_hand'], 1, 0)

# Multiply PX by -1 for all rows because trackman and trumedia are opposite
merged_df['PX'] = merged_df['PX'] * -1

# Multiply PX by -1 where pitcher_hand is 'L'
merged_df['PX'] = np.where(merged_df['pitcher_hand'] == 'L', merged_df['PX'] * -1, merged_df['PX'])

# ... existing code ...
merged_df.head(111)



In [ ]:
# 2) Plot the mph_adjustment results
plot_mph_adjustment_scatter(
    merged_df, 
    px_col='PX', 
    pz_col='PZ'
)


In [ ]:

# ... existing code ...

# Filter out unwanted pitch results before model training
# First remove exact match for 'In Play'


# Continue with your existing feature selection
features = [
    "start_speed",
    "spin_rate",
    "extension",
    "az",
    "ax",
    "x0",
    "z0", 
    "PX", 
    "PZ",
]
# ... existing code ...

In [ ]:
import pandas as pd
import numpy as np
import joblib

# Models
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Stacking & Ensemble
from sklearn.ensemble import StackingRegressor, RandomForestRegressor

# Train/Test Split, Metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# -------------------------------------------------------
# 1) PREPARE DATA
# -------------------------------------------------------
# Assume you have a DataFrame called merged_df with ~450k rows,
# and you want to predict "target" from a set of features.
features = [
    "start_speed",
    "spin_rate",
    "extension",
    "az",
    "ax",
    "x0",
    "z0",
    "PX",
    "PZ",
    'same_side'
]
target = "target"



merged_df = merged_df[merged_df['pitchResult'] != 'In Play']

# Then filter out rows containing other unwanted terms
unwanted_results = ['hit by pitch', 'interference', 'intentional', 'unknown']
merged_df = merged_df[~merged_df['pitchResult'].str.contains('|'.join(unwanted_results), case=False, na=False)]

# Drop rows with missing feature/target values
df_train = merged_df.dropna(subset=features + [target])




X = df_train[features]
y = df_train[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------------------------------
# 2) DEFINE BASE MODELS (GBDTs)
# -------------------------------------------------------
# You can add your preferred hyperparameters here:
lgbm_reg = LGBMRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31
)

xgb_reg = XGBRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    tree_method="auto"
)

cat_reg = CatBoostRegressor(
    random_seed=42,
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=0  # turn off catboost training logs
)

# -------------------------------------------------------
# 3) DEFINE STACKING REGRESSOR
# -------------------------------------------------------
stack_ensemble = StackingRegressor(
    estimators=[
        ("lgbm", lgbm_reg),
        ("xgb", xgb_reg),
        ("cat", cat_reg)
    ],
    final_estimator=RandomForestRegressor(
        n_estimators=200,
        max_depth=6,
        random_state=42
    ),
    # passthrough=True,  # optionally pass original features to final estimator
    cv=5  # number of cross-validation folds to blend base learners
)

# -------------------------------------------------------
# 4) TRAIN STACKED MODEL
# -------------------------------------------------------
stack_ensemble.fit(X_train, y_train)

# -------------------------------------------------------
# 5) EVALUATE ON TEST SET
# -------------------------------------------------------
y_pred = stack_ensemble.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {mse:.5f}")
print(f"Test R^2:  {r2:.5f}")

# -------------------------------------------------------
# 6) SAVE THE STACKED MODEL
# -------------------------------------------------------
joblib.dump(stack_ensemble, "rv_with_plateloc.joblib")
print("Stacked model saved to 'stacked_whiff_model.joblib'")


In [ ]:
%pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from matplotlib.patches import Rectangle, Polygon
import os

# 1) Create a dummy 'filtered_data' with random values
np.random.seed(42)
data = {
    "Pitcher": ["Pitcher1"] * 20,
    "Relspeed": np.random.uniform(90, 92, 20),
    "Spinrate": np.random.uniform(1800, 2200, 20),
    "Extension": np.random.uniform(5, 5.2, 20),
    "Relheight": np.random.uniform(5, 5.5, 20),
    "Relside": np.random.uniform(1, 1.1, 20),
    "Horzbreak": np.random.uniform(6, 7, 20),
    "Inducedvertbreak": np.random.uniform(16, 18, 20),
    "Platelocside": np.random.uniform(-2, 2, 20),
    "Platelocheight": np.random.uniform(1.5, 3.5, 20),
    "Pitchtype": ["Fastball"] * 20
}
filtered_data = pd.DataFrame(data)

# 2) Compute average relside and assign pitcher_hand
filtered_data["avg_relside"] = filtered_data.groupby("Pitcher")["Relside"].transform("mean")
filtered_data["pitcher_hand"] = np.where(filtered_data["avg_relside"] < 0, "L", "R")

# 3) Rename raw columns to model-ready names
filtered_data.rename(columns={
    "Relspeed":  "start_speed",
    "Spinrate":  "spin_rate",
    "Extension": "extension",
    "Relheight": "z0",
    "Relside":   "x0",
    "Horzbreak": "ax",
    "Inducedvertbreak": "az"
}, inplace=True)

# 4) Map plate location columns
filtered_data["PX"] = filtered_data["Platelocside"]
filtered_data["PZ"] = filtered_data["Platelocheight"]

# 5) Flip columns for left-handed pitchers
filtered_data.loc[filtered_data["pitcher_hand"] == "L", ["x0", "ax", "PX"]] *= -1

# 6) Optional: Convert z0 & x0 to inches if needed
filtered_data["z0"] = filtered_data["z0"] * 12
filtered_data["x0"] = filtered_data["x0"] * 12

# 7) Load your actual model
#    Make sure the path below is correct and the file is accessible.
model_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\Python Scripts\rv_with_plateloc.joblib"
rv_model = joblib.load(model_path)

# 8) Define grid parameters
px_min  = -2
px_max  =  2
px_step =  0.05
pz_min  =  0.5
pz_max  =  4
pz_step =  0.05

# 9) Required features for your model
required_features = [
    "start_speed", "spin_rate", "extension",
    "az", "ax", "x0", "z0", "PX", "PZ", "same_side"
]

def simulate_pitch_grid(pitch_type: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build a grid of PX, PZ, plus the median of other features from filtered_data,
    then split into same_side=1 and same_side=0, predict 'run_value' using your model,
    and return both DataFrames.
    """
    sub = filtered_data[filtered_data["Pitchtype"] == pitch_type].copy()
    if sub.empty:
        return pd.DataFrame(), pd.DataFrame()
    
    # Exclude same_side to compute medians
    features_for_medians = [f for f in required_features if f != "same_side"]
    medians = sub[features_for_medians].median(numeric_only=True).to_dict()

    # Create the PX/PZ grid
    px_values = np.arange(px_min, px_max + px_step, px_step)
    pz_values = np.arange(pz_min, pz_max + pz_step, pz_step)
    grid_rows = []
    for px in px_values:
        for pz in pz_values:
            row = medians.copy()
            row["PX"] = px
            row["PZ"] = pz
            grid_rows.append(row)
    grid_df = pd.DataFrame(grid_rows)

    # Split into same_side & opposite_side
    df_same = grid_df.copy()
    df_same["same_side"] = 1
    df_opposite = grid_df.copy()
    df_opposite["same_side"] = 0

    # Predict run_value
    df_same["run_value"] = rv_model.predict(df_same[required_features])
    df_opposite["run_value"] = rv_model.predict(df_opposite[required_features])

    # For testing: label pitcher_hand in both
    # (We assume the entire DataFrame is one pitcher for testing.)
    pitcher_hand_value = filtered_data["pitcher_hand"].iloc[0]
    df_same["pitcher_hand"] = pitcher_hand_value
    df_opposite["pitcher_hand"] = pitcher_hand_value

    # Flip PX for left-handed pitchers
    df_same.loc[df_same["pitcher_hand"] == "L", "PX"] *= -1
    df_opposite.loc[df_opposite["pitcher_hand"] == "L", "PX"] *= -1

    return df_same, df_opposite

# 10) Simulate for a single pitch type (e.g. "Fastball")
pitch_type = "Fastball"
df_same, df_opposite = simulate_pitch_grid(pitch_type)

if df_same.empty or df_opposite.empty:
    print("No data available for pitch type:", pitch_type)
    exit()

# 11) Plot side-by-side heatmaps using Seaborn
pitcher_hand = filtered_data["pitcher_hand"].iloc[0]
opposite_hand = "R" if pitcher_hand == "L" else "L"

# Plot both heatmaps side by side using scatterplots instead of pivoting.
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Plot same-side data
sns.scatterplot(data=df_same, x="PX", y="PZ", hue="run_value", palette="RdBu_r", ax=axes[0])
axes[0].invert_yaxis()  # so higher PZ values are at the top
axes[0].set_title(f"Vs. {pitcher_hand}HH")

# Plot opposite-side data
sns.scatterplot(data=df_opposite, x="PX", y="PZ", hue="run_value", palette="RdBu_r", ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title(f"Vs. {opposite_hand}HH")

# Add the strike zone rectangle and home plate polygon.
plate_vertices = [(-0.83, 0.1), (0.83, 0.1), (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]
for ax in axes:
    ax.add_patch(Rectangle((-0.83, 1.5), 1.66, 2.1, edgecolor='black', facecolor='none'))
    plate = Polygon(plate_vertices, closed=True, linewidth=1, edgecolor='k', facecolor='none')
    ax.add_patch(plate)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5)

plt.tight_layout()
plt.show(fig)

In [ ]:
df_same.head()

In [ ]:
grid_df.head(111)